# Suite DOM — Value domains

A literal carries its value domain: a standard's representation (`Integer`, `Ieee754`, `Bits`, `Bytes`, `Unicode`,
`Ieee1164`, `Enum`, `Packed`, `Bool`), or by default its native's. Domains are data, compared by structure and
validated; a registered domain is written by name, any other by value, and a default one not at all. Evaluating
literals of other domains comes in later phases.

In [ ]:
import { Domains as D, Evaluators as V, Expressions as E } from "@mbse/expressions";
import * as Python from "@mbse/expressions/Dialects/Python";
import { Domains as FD, Terms as F } from "@mbse/expressions/Framework";
import * as Translators from "@mbse/expressions/Translators";
import { Comparison as C, Errors, JSON, Plain, Validators, YAML } from "@mbse/schemas/Framework";

const { LookupError, NotImplementedError, ValueError } = Errors;
import { assert, equal, map, raises, same } from "./support.js";

const I8 = new D.OfInteger.Builder().width(8n).create();
const U8 = new D.OfInteger.Builder().width(8n).signed(false).overflow("wrap").create();
const F32 = new D.OfIeee754.Builder().format("binary32").create();
const Bits2 = new D.OfBits.Builder().width(2n).create();
const State = new D.OfEnum.Builder().members("IDLE", "RUN").create();
const Coded = new D.OfPacked.Builder().domain(State).representation(Bits2).codes(0n, 1n).create();

/** A literal's snapshot, as plain data. */
function snapshot(literal: unknown): any {
  return (Plain.ToPlain(E.OfLiteral.Schema, literal as never) as any).get("objects").get("s0");
}

## DOM-01 · Value domains name standards, with only their parameters, are compared by structure and validated

In [ ]:
{
  const names: [D.Domain, string][] = [[D.Bool, "bool"], [D.Int, "int"], [I8, "int8"], [U8, "uint8"],
    [new D.OfInteger.Builder().signed(false).create(), "uint"], [D.Float, "float"], [F32, "binary32"],
    [new D.OfIeee754.Builder().rounding("roundTowardZero").create(), "binary64 roundTowardZero"],
    [Bits2, "bits2"], [D.Bytes, "bytes"], [new D.OfBytes.Builder().width(4n).create(), "bytes4"], [D.Str, "str"],
    [new D.OfIeee1164.Builder().create(), "std_logic"], [State, "enum(IDLE, RUN)"], [Coded, "packed(enum(IDLE, RUN) as bits2)"]];
  assert(names.every(([domain, name]) => domain.name() === name && String(domain) === name));
  assert(names.every(([domain]) => domain.validate().length === 0));
  // The natives' defaults, compared by structure: an equal domain built anew is the same domain.
  assert(new D.OfInteger.Data().equals(D.Int) && new D.OfIeee754.Data("binary64").equals(D.Float) && new D.OfUnicode.Data().equals(D.Str));
  assert(new D.OfBytes.Data().equals(D.Bytes) && new D.OfBool.Data().equals(D.Bool) && !I8.equals(U8) && !I8.equals(D.Int));
  assert(new D.OfInteger.Builder().width(8n).create().equals(I8) && String(D.OfInteger) === "<domain kind integer>");
  assert([D.OfBool, D.OfInteger, D.OfIeee754, D.OfBits, D.OfBytes, D.OfUnicode, D.OfIeee1164, D.OfEnum, D.OfPacked]
    .every((kind) => kind.Schema.validate().length === 0) && D.Schema.validate().length === 0);
  assert(same(D.Schema.branches.map((b) => b.name), ["bool", "integer", "ieee754", "bits", "bytes", "unicode", "ieee1164",
    "enum", "packed", "named"]));
  assert(D.FORMATS[2] === "binary64" && D.ROUNDINGS[0] === "roundTiesToEven" && same(D.OVERFLOWS, ["wrap", "saturate", "raise"]));
  assert(D.STATES.length === 9);
  // Each kind's problems.
  const problems: [D.Domain, string[]][] = [
    [new D.OfInteger.Data(0n), ["a width must be a positive int, got 0"]],
    [new D.OfInteger.Data(8n, "yes" as never, "clip"), ["signed must be a bool, got 'yes'",
      "overflow must be one of wrap, saturate, raise, got 'clip'"]],
    [new D.OfIeee754.Data("binary8", "up"), [
      "format must be one of binary16, binary32, binary64, binary128, decimal64, decimal128, got 'binary8'",
      "rounding must be one of roundTiesToEven, roundTiesToAway, roundTowardPositive, roundTowardNegative, " +
      "roundTowardZero, got 'up'"]],
    [new D.OfBits.Data(), ["a bits domain needs a width"]],
    [new D.OfBits.Data(0n), ["a width must be a positive int, got 0"]],
    [new D.OfBytes.Data(-1n), ["a width must be a positive int, got -1"]],
    [new D.OfEnum.Data(), ["an enum needs a member"]],
    [new D.OfEnum.Data(["A", "", "A"]), ["member 1 has no name", "member 'A' appears twice"]],
    [new D.OfPacked.Data(I8, Bits2, []), ["a packed domain packs an enum, got int8"]],
    [new D.OfPacked.Data(), ["a packed domain packs an enum, got None"]],
    [new D.OfPacked.Data(State, D.Bytes, [0n, 1n]), ["a packed domain's representation is an integer or bits of a width, got bytes"]],
    [new D.OfPacked.Data(State, D.Int, [0n, 1n]), ["a packed domain's representation is an integer or bits of a width, got int"]],
    [new D.OfPacked.Data(State, Bits2, [0n]), ["a packed domain needs a code per member: 2 members, 1 codes"]],
    [new D.OfPacked.Data(State, new D.OfBits.Data(1n), [0n, 2n]), ["code 1 does not fit bits1: 2"]],
    [new D.OfPacked.Data(State, Bits2, [1n, 1n]), ["code 1 is used twice"]],
    [new D.OfPacked.Data(State, new D.OfInteger.Data(2n), [-2n, 2n]), ["code 1 does not fit int2: 2"]],
    [new D.OfPacked.Data(State, new D.OfInteger.Data(2n), ["0" as never, 1n]), ["code 0 does not fit int2: '0'"]],
    [new D.OfPacked.Data(new D.OfEnum.Data(), new D.OfInteger.Data(2n, true, "x"), []), [
      "domain: an enum needs a member", "representation: overflow must be one of wrap, saturate, raise, got 'x'"]],
  ];
  for (const [domain, expected] of problems) assert(same(domain.validate(), expected), `${domain}: ${domain.validate()}`);
  assert(new D.OfPacked.Data().name() === "packed(None as None)");
  // Builders finalize as every builder does, and set each parameter.
  const built = new D.OfIeee754.Builder(F32).rounding("roundTowardNegative").clone();
  assert(built.equals(new D.OfIeee754.Data("binary32", "roundTowardNegative")) && F32.rounding === "roundTiesToEven");
  assert(new D.OfBytes.Builder(new D.OfBytes.Data()).width(3n).update().width === 3n);
  for (const builder of [new D.OfBool.Builder(), new D.OfUnicode.Builder(), new D.OfIeee1164.Builder()]) {
    assert(builder.create().validate().length === 0);
  }
  raises(ValueError, () => new D.OfInteger.Builder(I8).create(), "create() is only valid without a source instance; use clone() or update()");
  raises(ValueError, () => new D.OfInteger.Builder().clone(), "clone() is only valid with a source instance");
  raises(ValueError, () => new D.OfInteger.Builder().update(), "update() is only valid with a source instance");
  raises(TypeError, () => new D.OfInteger.Builder(Bits2 as never), "expected integer data to build from, got bits2");
}

## DOM-02 · A domain holds the values of its native type that fit it, and includes only itself

In [ ]:
{
  const bytes = (...values: number[]) => new Uint8Array(values);
  const holds: [D.Domain, unknown[], unknown[]][] = [
    [I8, [127n, -128n], [128n, -129n, true, 1.0]], [U8, [0n, 255n], [-1n, 256n]], [D.Int, [10n ** 30n, -5n], [false]],
    [new D.OfInteger.Data(null, false), [10n ** 30n], [-1n]], [new D.OfInteger.Data(0n), [10n ** 30n], []], // not a width: unbounded
    [F32, [1.5, NaN], [1n, "1.5"]], [new D.OfIeee754.Data("decimal64"), ["0.1"], [0.1]],
    [new D.OfBits.Data(3n), [bytes(7), bytes(0)], [bytes(8), bytes(0, 7), "7"]], [new D.OfBits.Data(8n), [bytes(255)], [bytes()]],
    [new D.OfBits.Data(9n), [bytes(1, 255)], [bytes(2, 0), bytes(255)]], [new D.OfBits.Data(), [], [bytes(0)]],
    [new D.OfBytes.Data(2n), [bytes(97, 98)], [bytes(97)]], [D.Bytes, [bytes(), bytes(97, 98, 99)], ["abc"]],
    [D.Str, ["", "日本語 🚀"], ["\ud800", bytes(97)]], [new D.OfIeee1164.Data(), ["Z", "-", "U"], ["z", "2", 1n]],
    [State, ["RUN"], ["STOP", 0n]], [Coded, ["IDLE"], ["STOP"]], [new D.OfPacked.Data(), [], ["IDLE"]],
    [D.Bool, [true, false], [1n, 0n]],
  ];
  for (const [domain, inside, outside] of holds) {
    assert(inside.every((v) => domain.contains(v)) && !outside.some((v) => domain.contains(v)), String(domain));
  }
  assert(new D.OfIeee754.Data("decimal128").native() === String && F32.native() === Number && I8.native() === BigInt);
  // A domain includes only domains equal to it: Basic has no implied promotions.
  assert(I8.includes(new D.OfInteger.Data(8n)) && !I8.includes(D.Int) && !D.Int.includes(I8) && !I8.includes(U8));
  assert(I8.includes(new FD.OfUnion(I8, new D.OfInteger.Data(8n))) && !I8.includes(new FD.OfUnion(I8, U8)));
  assert(FD.overlaps(FD.Anything, I8) && !D.Float.includes(F32));
  // A native's default domain, and the domains of other values.
  assert(same([true, 1n, 1.5, "a", bytes(97)].map((v) => D.of(v)), [D.Bool, D.Int, D.Float, D.Str, D.Bytes]));
  assert(D.of(E.variable("x").data) === D.Object);
  raises(TypeError, () => D.of(null), "a NoneType is not a value of the Basic dialect");
}

## DOM-03 · Domains are registered by name, once, and written by name; any other by value

In [ ]:
{
  D.register("uint8", U8);
  assert(D.registered("uint8") === U8 && D.name_of(U8) === "uint8" && D.name_of(new D.OfInteger.Data(8n, false, "wrap")) === "uint8");
  assert(D.name_of(I8) === null);
  raises(ValueError, () => D.register("uint8", I8), "domain 'uint8' is already registered");
  raises(ValueError, () => D.register("byte", new D.OfInteger.Data(8n, false, "wrap")), "uint8 is already registered as 'uint8'");
  D.register("uint16", new D.OfInteger.Data(16n, false)); // another domain: each name, and each domain, once
  assert(D.name_of(new D.OfInteger.Data(16n, false)) === "uint16");
  raises(LookupError, () => D.registered("nope"), "no domain registered as 'nope'");
  assert(equal(D.to_plain(U8), map({ named: { name: "uint8" } })));
  assert(equal(D.to_plain(I8), map({ integer: { width: 8n, signed: true, overflow: "raise" } })));
  assert(equal(D.to_plain(D.Int), map({ integer: { signed: true, overflow: "raise" } })));
  assert(equal(D.to_plain(new D.OfPacked.Data(State, U8, [0n, 1n])), map({
    packed: { domain: { enum: { members: ["IDLE", "RUN"] } }, representation: { named: { name: "uint8" } }, codes: [0n, 1n] } })));
  assert(equal(D.to_plain(new D.OfUnicode.Data()), map({ unicode: {} })));
  for (const domain of [U8, I8, F32, Bits2, D.Bytes, D.Str, State, Coded, new D.OfPacked.Data(State, U8, [0n, 1n]), D.Bool]) {
    assert(D.from_plain(D.to_plain(domain)).equals(domain), String(domain));
  }
  raises(TypeError, () => D.to_plain("int8"), "not a domain: 'int8'");
  raises(LookupError, () => D.from_plain(map({ named: { name: "nope" } })), "no domain registered as 'nope'");
}

## DOM-04 · A literal carries its domain; a native's default domain is no domain at all

In [ ]:
{
  const lit = E.literal(200n, U8).data as E.OfLiteral.Data;
  assert(lit.domain === U8 && lit.typed() === U8 && lit.validate().length === 0);
  assert((E.literal(5n, D.Int).data as E.OfLiteral.Data).domain === null && E.literal(5n).data.typed() === null); // the default is left out
  assert((new E.OfLiteral.Builder().value(-1n).domain(I8).create().domain as D.Domain).equals(I8));
  assert(new E.OfLiteral.Builder(lit).value(7n).clone().domain === U8 && new E.OfLiteral.Builder(lit).domain(null).clone().domain === null);
  assert(new E.OfLiteral.Builder(E.literal(1n).data as E.OfLiteral.Data).domain(D.Int).update().domain === null);
  // A literal's domain is valid and holds its value.
  assert(same(E.literal(300n, U8).data.validate(), ["a literal of uint8 cannot hold 300"]));
  assert(same(E.literal(new Uint8Array([0]), new D.OfBits.Data()).data.validate(), ["domain: a bits domain needs a width"]));
  assert(same(new E.OfLiteral.Data(1n, "int8").validate(), ["a literal's domain must be a value domain, got str"]));
  assert(same(E.operation("eq", E.literal("RUN", Coded), E.literal("STOP", Coded)).data.validate(),
    ["argument 1: a literal of packed(enum(IDLE, RUN) as bits2) cannot hold 'STOP'"]));
  // Inference gives a literal's domain, and Basic does not promote: int8 and int8 add only once arithmetic takes them.
  assert(E.DIALECT.infer(lit) === U8 && E.DIALECT.infer(E.literal(1n)) === D.Int);
  raises(TypeError, () => E.DIALECT.infer(E.operation("add", E.literal(1n, I8), E.literal(2n, I8))),
    "add cannot take (int8, int8); it takes (T, T) -> T for T in int | float");
  // Forms, `make` and `same` carry the domain.
  const attributes = lit.form().attributes;
  assert(attributes.size === 2 && attributes.get("value") === 200n && attributes.get("domain") === U8);
  assert(F.same(E.DIALECT.make(lit.form()), lit) && F.same(lit, E.literal(200n, new D.OfInteger.Data(8n, false, "wrap")).data));
  assert(!F.same(lit, E.literal(200n, I8).data) && !F.same(lit, E.literal(200n).data));
  // Evaluation of other domains comes later; a default domain evaluates as before.
  raises(NotImplementedError, () => V.OfAny(lit), "literals of uint8 are not evaluated yet");
  assert(V.OfAny(E.operation("add", E.literal(1n, D.Int), 2n)) === 3n);
  // Translation keeps a domain only where the other dialect has one: none does yet.
  raises(ValueError, () => Translators.between(E.DIALECT, Python.Expressions.DIALECT).forward(lit),
    "Basic literal 200 of uint8 has no Python counterpart");
}

## DOM-05 · Over the wire a literal's domain is written by name when registered, otherwise by value

In [ ]:
{
  assert(equal(snapshot(E.literal(200n, U8).data), map({ kind: "literal", int: 200n, domain: { named: { name: "uint8" } } })));
  assert(equal(snapshot(E.literal(-1n, I8).data), map({
    kind: "literal", int: -1n, domain: { integer: { width: 8n, signed: true, overflow: "raise" } } })));
  assert(equal(snapshot(E.literal(5n, D.Int).data), map({ kind: "literal", int: 5n })));
  const text = JSON.ToJSON(E.OfLiteral.Schema, E.literal("RUN", Coded).data as never);
  assert(text === '{"root": "s0", "objects": {"s0": {"kind": "literal", "str": "RUN", "domain": {"packed": {"domain": ' +
    '{"enum": {"members": ["IDLE", "RUN"]}}, "representation": {"bits": {"width": 2}}, "codes": [0, 1]}}}}}');
  for (const literal of [E.literal(200n, U8).data, E.literal(-1n, I8).data, E.literal("RUN", Coded).data, E.literal(1.5, F32).data,
    E.literal("Z", new D.OfIeee1164.Data()).data, E.literal(new Uint8Array([3]), Bits2).data] as E.OfLiteral.Data[]) {
    const back = JSON.FromJSON(E.Builders as never)(E.OfLiteral.Schema, JSON.ToJSON(E.OfLiteral.Schema, literal as never)) as E.OfLiteral.Data;
    assert(F.same(back, literal) && (back.domain as D.Domain).equals(literal.domain));
    assert(F.same(YAML.FromYAML(E.Builders as never)(E.OfLiteral.Schema, YAML.ToYAML(E.OfLiteral.Schema, literal as never)), literal));
    assert(Validators.Validate(E.Builders as never)(E.OfLiteral.Schema, back).length === 0);
  }
  assert((JSON.FromJSON(E.Builders as never)(E.OfLiteral.Schema, JSON.ToJSON(E.OfLiteral.Schema, E.literal(1n, U8).data as never)) as
    E.OfLiteral.Data).domain === U8);
  raises(LookupError, () => Plain.FromPlain(E.Builders as never)(E.OfLiteral.Schema, map({ root: "s0", objects: { s0: {
    kind: "literal", int: 1n, domain: { named: { name: "nope" } } } } })), "no domain registered as 'nope'");
  // Literals compare under their meta-schema with their domains.
  const compare = (a: unknown, b: unknown) => new C.OfObject(E.OfLiteral.Schema, a as never).compare(new C.OfObject(E.OfLiteral.Schema, b as never));
  assert(compare(E.literal(1n, I8).data, E.literal(1n, new D.OfInteger.Data(8n)).data) === 0);
  assert(compare(E.literal(1n, I8).data, E.literal(1n, U8).data) === null && compare(E.literal(1n, I8).data, E.literal(1n).data) === null);
  // The builder reads and writes the domain through the visitor protocols too.
  const domainField = (builder: any, branch: string, name: string, act: (n: any) => unknown) =>
    builder.property("domain", (p: any) => p.value((a: any) => a.as_union((u: any) => u.property(branch, (q: any) => q.value(
      (b: any) => b.as_object((o: any) => o.property(name, (w: any) => w.value((n: any) => n.as_native(act)))))))));
  const builder = new E.OfLiteral.Builder(E.literal(-1n, I8).data as E.OfLiteral.Data);
  const names: string[] = [];
  builder.properties((p) => names.push(p.name()));
  assert(builder.has("domain") && same(names, ["kind", "int", "domain"]));
  const widths: unknown[] = [];
  domainField(builder, "integer", "width", (n) => widths.push(n.get()));
  assert(same(widths, [8n]));
  domainField(builder, "integer", "width", (n) => n.set(16n));
  assert((builder.clone().domain as D.Domain).equals(new D.OfInteger.Data(16n)));
  builder.property("domain", (p) => p.clear());
  assert(!builder.has("domain") && builder.clone().domain === null);
  domainField(builder, "named", "name", (n) => n.set("uint8"));
  assert(builder.clone().domain === U8);
  builder.clear("domain");
  assert(builder.clone().domain === null);
  builder.property("domain", (p) => p.value((a) => a.as_union(() => null))); // a union without a branch is no value
  assert(builder.clone().domain === null);
}